# Model training

Build and evaluate credit-default classification models while keeping the final test set isolated. Start with an explicit Logistic Regression baseline, then use five-fold cross-validation for model selection and tuning.

The notebook keeps preprocessing steps visible for peer review: categorical encoding and numerical scaling are learned only from the relevant training data.

In [53]:
import sys
from pathlib import Path


root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

sys.path.insert(0, str(root / "code"))

from paths import RAW_DATA_FILE

print(RAW_DATA_FILE)
print("Dataset exists:", RAW_DATA_FILE.is_file())

/workspaces/credit_defaulf_prediction/data/default_of_credit_card_clients.xls
Dataset exists: True


In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer

## Load the dataset

Open the original Excel workbook and inspect its layout before loading the modeling table. The descriptive column names are stored in the second spreadsheet row, so `header=1` is required.

In [55]:
excel = pd.ExcelFile(RAW_DATA_FILE, engine="xlrd")
print("Sheets:", excel.sheet_names)

preview = pd.read_excel(excel, sheet_name=0, header=None, nrows=8)
display(preview)

DF = pd.read_excel(excel, sheet_name=0, header=1)

print("Shape:", DF.shape)
display(DF.head())
print("Columns:", DF.columns.tolist())

/tmp/ipykernel_70756/2519810770.py:1: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  excel = pd.ExcelFile(RAW_DATA_FILE, engine="xlrd")


Sheets: ['Data']


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
1,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
2,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
3,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
4,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
5,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
6,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
7,6,50000,1,1,2,37,0,0,0,0,...,19394,19619,20024,2500,1815,657,1000,1000,800,0


Shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


Columns: ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


### Dataset check

The workbook contains one `Data` sheet and loads 30,000 records with 25 columns: an ID, 23 predictors, and the binary target. Normalize the column names before selecting model inputs.

## Prepare modeling columns

Normalize column names to lowercase snake case, identify `default_payment_next_month` as the target, and define the 23 predictors. Exclude `id` because it identifies rows rather than describing customer behavior.

In [56]:
DF.columns = DF.columns.str.strip().str.lower().str.replace(" ", "_")

In [57]:
target_cols = ['default_payment_next_month']
feature_cols = [col for col in DF.columns if col not in ['default_payment_next_month','id'] ]



### Modeling column results

The target is stored separately from the 23 predictors, and `id` is excluded from model inputs. Define shared split parameters next so that every random operation can be reproduced.

In [58]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

## Split the data

First reserve 20% of the complete dataset as the final test set. Split the remaining 80% into a 60% training set and a 20% validation set for the initial baseline. The complete 80% `df_full_train` is retained for stratified cross-validation and tuning.

Use the same random seed for reproducibility. The test set must remain isolated until the final model has been selected.

In [59]:
df_full_train, df_test = train_test_split(
    DF,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

validation_size = TEST_SIZE / (1 - TEST_SIZE)
df_train, df_val = train_test_split(
    df_full_train,
    test_size=validation_size,
    random_state=RANDOM_STATE,
)

split_data = {
    "full_train": df_full_train,
    "train": df_train,
    "validation": df_val,
    "test": df_test,
}

display(pd.DataFrame({
    "records": [len(part) for part in split_data.values()],
    "pct_size": [len(part) / len(DF) for part in split_data.values()],
    "default_rate": [
        part["default_payment_next_month"].mean()
        for part in split_data.values()
    ],
}, index=split_data.keys()))

,records,pct_size,default_rate
full_train,24000,0.8,0.221792
train,18000,0.6,0.222611
validation,6000,0.2,0.219333
test,6000,0.2,0.218833


### Split results

The split produces 24,000 full-training records and 6,000 test records. The full-training data is also divided into 18,000 baseline-training records and 6,000 validation records. Default rates range from 21.88% to 22.26%, so the random split preserves a similar class balance across all subsets.

## Separate features and target

Create feature matrices and target vectors after splitting the DataFrame. Exclude both the row identifier and the target from the 23 model inputs, preserve row indices, and verify that the subsets do not overlap.

In [60]:
target = target_cols[0]

X_train = df_train[feature_cols].copy()
y_train = df_train[target].copy()

X_val = df_val[feature_cols].copy()
y_val = df_val[target].copy()

X_test = df_test[feature_cols].copy()
y_test = df_test[target].copy()

X_full_train = df_full_train[feature_cols].copy()
y_full_train = df_full_train[target].copy()

for name, X, y in [
    ("full_train", X_full_train, y_full_train),
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
]:
    assert X.index.equals(y.index)
    assert X.shape[1] == 23
    assert "id" not in X.columns
    assert target not in X.columns
    print(f"{name}: X={X.shape}, y={y.shape}")

assert df_train.index.intersection(df_val.index).empty
assert df_train.index.intersection(df_test.index).empty
assert df_val.index.intersection(df_test.index).empty
assert df_full_train.index.intersection(df_test.index).empty
assert set(df_full_train.index) == set(df_train.index).union(df_val.index)

full_train: X=(24000, 23), y=(24000,)
train: X=(18000, 23), y=(18000,)
validation: X=(6000, 23), y=(6000,)
test: X=(6000, 23), y=(6000,)


### Feature and target results

The full-training, baseline-training, validation, and test feature matrices each contain the expected 23 predictors. Their target vectors preserve the same row indices, and the index checks confirm that the final test set does not overlap the development data.

## Encode categorical features

Treat `sex`, `education`, and `marriage` as categorical variables. Group the undocumented education codes 5 and 6 with category 0, while preserving marriage category 0 as unknown.

Fit `OneHotEncoder` only on baseline-training data and reuse it to transform validation data. `handle_unknown="ignore"` prevents an unseen validation category from causing an error.

In [61]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ["sex", "education", "marriage"]

X_train_prepared = X_train.copy()
X_val_prepared = X_val.copy()

# Group undocumented education codes into category 0.
for subset in [X_train_prepared, X_val_prepared]:
    subset["education"] = subset["education"].replace({5: 0, 6: 0})

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

# Learn categories from training data only.
train_categories = encoder.fit_transform(
    X_train_prepared[categorical_cols]
)
val_categories = encoder.transform(
    X_val_prepared[categorical_cols]
)

print("Encoded columns:", encoder.get_feature_names_out())
print("Train:", train_categories.shape)
print("Validation:", val_categories.shape)

Encoded columns: ['sex_1' 'sex_2' 'education_0' 'education_1' 'education_2' 'education_3'
 'education_4' 'marriage_0' 'marriage_1' 'marriage_2' 'marriage_3']
Train: (18000, 11)
Validation: (6000, 11)


### Encoding results

The three categorical inputs produce 11 binary columns: 2 for sex, 5 for education, and 4 for marriage. Training and validation use the same encoded feature names, while the original DataFrames remain unchanged.

## Combine numerical and encoded features

Combine the 20 numerical features with the 11 columns produced by One-Hot Encoding, creating 31 model inputs. Preserve the original row indices so that each customer's features remain aligned with the correct target. Check that training and validation have the same columns and no missing values introduced by concatenation.

In [62]:
numeric_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

encoded_cols = encoder.get_feature_names_out(categorical_cols)

train_encoded = pd.DataFrame(
    train_categories,
    columns=encoded_cols,
    index=X_train_prepared.index,
)

val_encoded = pd.DataFrame(
    val_categories,
    columns=encoded_cols,
    index=X_val_prepared.index,
)

X_train_encoded = pd.concat(
    [X_train_prepared[numeric_cols], train_encoded],
    axis=1,
)

X_val_encoded = pd.concat(
    [X_val_prepared[numeric_cols], val_encoded],
    axis=1,
)

assert X_train_encoded.index.equals(y_train.index)
assert X_val_encoded.index.equals(y_val.index)
assert X_train_encoded.columns.equals(X_val_encoded.columns)
assert not X_train_encoded.isna().any().any()
assert not X_val_encoded.isna().any().any()

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)

Train: (18000, 31)
Validation: (6000, 31)


### Combined feature results

The 20 numerical inputs and 11 encoded categorical inputs produce 31 model features. Training and validation retain their original row counts and matching columns, with no missing values introduced during concatenation.

## Scale numerical features

Standardize the 20 numerical features using StandardScaler. This brings variables with different units and ranges to comparable scales, helping Logistic Regression converge and making regularization less dependent on feature scale. Keep the One-Hot columns unchanged at 0/1.

Learn means and standard deviations from training data only, then apply the same transformation to validation data to avoid leakage. The training means are approximately zero; the observed residual of 1.39e-16 is normal floating-point rounding.

In [63]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_val_scaled = X_val_encoded.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(
    X_train_encoded[numeric_cols]
)

X_val_scaled[numeric_cols] = scaler.transform(
    X_val_encoded[numeric_cols]
)

print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)

print(
    "Largest absolute numerical mean:",
    X_train_scaled[numeric_cols].mean().abs().max(),
)

Train: (18000, 31)
Validation: (6000, 31)
Largest absolute numerical mean: 5.2303840271229595e-17


### Scaling results

Scaling preserves the 18,000 × 31 training matrix and the 6,000 × 31 validation matrix. The largest absolute mean among scaled training variables is approximately 5.23e-17, effectively zero; the tiny residual is normal floating-point rounding.

## Logistic Regression baseline

Train a first Logistic Regression model on the encoded and scaled baseline-training data. Use predicted probabilities for the positive class (`default = 1`) to calculate ROC-AUC, which measures how well the model ranks clients by default risk across all possible classification thresholds.

Compare training and validation ROC-AUC for an initial overfitting check, and inspect the solver iteration count for convergence. Keep the test set untouched.

In [64]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

model.fit(X_train_scaled, y_train)

train_probabilities = model.predict_proba(X_train_scaled)[:, 1]
val_probabilities = model.predict_proba(X_val_scaled)[:, 1]

print("Train ROC-AUC:", roc_auc_score(y_train, train_probabilities))
print("Validation ROC-AUC:", roc_auc_score(y_val, val_probabilities))
print("Solver iterations:", model.n_iter_)

Train ROC-AUC: 0.7252286344962207
Validation ROC-AUC: 0.7198865730586437
Solver iterations: [66]


### Baseline results

The baseline reaches a ROC-AUC of 0.7252 on training data and 0.7199 on validation data. The gap is approximately 0.0053, which suggests limited overfitting on this split. The solver converges in 66 iterations, below the limit of 1,000, with no convergence warning. Cross-validation is still required to measure variation across different samples.

## Inspect five stratified cross-validation folds

Apply `StratifiedKFold` to all 24,000 records in `X_full_train`. Each iteration uses 19,200 records for fitting and 4,800 for validation. Shuffling makes the folds independent of the original row order, while the fixed seed makes them reproducible.

Defaults represent about 22% of the target. Stratification preserves approximately the same default rate in every fitting and validation fold, making model comparisons less sensitive to random differences in class composition. The final test set is not involved.

In [65]:
from sklearn.model_selection import StratifiedKFold

cv_splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_summary = []

for fold, (fit_idx, valid_idx) in enumerate(
    cv_splitter.split(X_full_train, y_full_train),
    start=1,
):
    fold_summary.append({
        "fold": fold,
        "train_records": len(fit_idx),
        "validation_records": len(valid_idx),
        "train_default_rate": y_full_train.iloc[fit_idx].mean(),
        "validation_default_rate": y_full_train.iloc[valid_idx].mean(),
    })

display(pd.DataFrame(fold_summary))

,fold,train_records,validation_records,train_default_rate,validation_default_rate
0,1,19200,4800,0.221823,0.221667
1,2,19200,4800,0.221823,0.221667
2,3,19200,4800,0.221771,0.221875
3,4,19200,4800,0.221771,0.221875
4,5,19200,4800,0.221771,0.221875


### Fold inspection results

Each fold has the expected 19,200 fitting records and 4,800 validation records. Stratification keeps the default rate approximately equal across folds, so differences in ROC-AUC are less affected by class-composition variation.

## Cross-validate the Logistic Regression baseline

Train and evaluate Logistic Regression once per fold using the 24,000 full-training records. Repeat category recoding, One-Hot Encoding, and numerical scaling inside each fold so that every validation fold is transformed only with information learned from its corresponding fitting fold.

Record validation ROC-AUC and solver iterations for every fold, then summarize the mean and standard deviation. Continue to keep the final test set isolated.

In [66]:
fold_results = []

for fold, (fit_idx, valid_idx) in enumerate(
    cv_splitter.split(X_full_train, y_full_train),
    start=1,
):
    # Select the records assigned to this fold.
    X_fit = X_full_train.iloc[fit_idx].copy()
    y_fit = y_full_train.iloc[fit_idx].copy()
    X_valid = X_full_train.iloc[valid_idx].copy()
    y_valid = y_full_train.iloc[valid_idx].copy()

    # Apply the documented education grouping to both subsets.
    X_fit["education"] = X_fit["education"].replace({5: 0, 6: 0})
    X_valid["education"] = X_valid["education"].replace({5: 0, 6: 0})

    # Learn categories only from this fold's fitting records.
    fold_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
    fit_categories = fold_encoder.fit_transform(X_fit[categorical_cols])
    valid_categories = fold_encoder.transform(X_valid[categorical_cols])
    fold_encoded_cols = fold_encoder.get_feature_names_out(categorical_cols)

    X_fit_encoded = pd.concat([
        X_fit[numeric_cols],
        pd.DataFrame(
            fit_categories,
            columns=fold_encoded_cols,
            index=X_fit.index,
        ),
    ], axis=1)

    X_valid_encoded = pd.concat([
        X_valid[numeric_cols],
        pd.DataFrame(
            valid_categories,
            columns=fold_encoded_cols,
            index=X_valid.index,
        ),
    ], axis=1)

    # Learn scaling parameters only from this fold's fitting records.
    fold_scaler = StandardScaler()
    X_fit_encoded[numeric_cols] = fold_scaler.fit_transform(
        X_fit_encoded[numeric_cols]
    )
    X_valid_encoded[numeric_cols] = fold_scaler.transform(
        X_valid_encoded[numeric_cols]
    )

    fold_model = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    fold_model.fit(X_fit_encoded, y_fit)

    valid_probabilities = fold_model.predict_proba(X_valid_encoded)[:, 1]

    fold_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_valid, valid_probabilities),
        "iterations": fold_model.n_iter_[0],
    })

cv_results = pd.DataFrame(fold_results)
display(cv_results)

print("Mean ROC-AUC:", cv_results["roc_auc"].mean())
print("ROC-AUC standard deviation:", cv_results["roc_auc"].std())

,fold,roc_auc,iterations
0,1,0.730490,59
1,2,0.723353,64
2,3,0.718163,66
3,4,0.725550,62
4,5,0.715010,66


Mean ROC-AUC: 0.7225134260189726
ROC-AUC standard deviation: 0.0060986602908979165


### Cross-validation results

Record the mean ROC-AUC, its fold-to-fold standard deviation, the score range, and whether every fold converged below the iteration limit. Compare the mean with the earlier fixed-validation baseline before tuning hyperparameters.

## Compare numerical and categorical encoding for repayment status

The six `pay_*` variables contain repayment-status codes. Treating them as numerical values imposes an order and distance between codes, while One-Hot Encoding treats every observed code as a separate state. The meanings of -2 and 0 are not documented by the consulted UCI source, so both representations remain plausible.

Compare two preprocessing strategies while keeping the dataset, stratified folds, Logistic Regression settings, and ROC-AUC scoring identical:

- **Numerical:** include `pay_*` among numerical variables and apply `StandardScaler`.
- **Categorical:** include `pay_*` among categorical variables and apply `OneHotEncoder`.

A pipeline is used so that category learning and scaling occur independently inside every fitting fold. The held-out test set remains untouched.

In [67]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


# Repayment status columns whose representation is being compared.
repayment_cols = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]

# These three variables remain categorical in both strategies.
base_categorical_cols = ["sex", "education", "marriage"]

# All remaining variables are quantitative amounts or age.
base_numerical_cols = [
    "limit_bal",
    "age",
    *[f"bill_amt{i}" for i in range(1, 7)],
    *[f"pay_amt{i}" for i in range(1, 7)],
]


def group_education_codes(X):
    """Group undocumented education codes without modifying input data."""
    X = X.copy()
    X["education"] = X["education"].replace({5: 0, 6: 0})
    return X


def build_logreg_pipeline(categorical_features, numerical_features):
    """Build one complete preprocessing and Logistic Regression workflow."""

    # ColumnTransformer applies a different transformation to each
    # explicitly listed group of input columns.
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_features,
            ),
            (
                "numerical",
                StandardScaler(),
                numerical_features,
            ),
        ],

        # Every model feature must belong to one of the two groups.
        remainder="drop",
    )

    # Pipeline executes each step in order. During cross-validation,
    # scikit-learn clones and fits a new pipeline inside every fold.
    return Pipeline(steps=[
        (
            "education_recoding",
            FunctionTransformer(
                group_education_codes,
                feature_names_out="one-to-one",
            ),
        ),
        ("preprocessing", preprocessing),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ])


# Strategy A: repayment status remains numerical and is scaled.
numerical_pipeline = build_logreg_pipeline(
    categorical_features=base_categorical_cols,
    numerical_features=base_numerical_cols + repayment_cols,
)

# Strategy B: repayment status is One-Hot Encoded and not scaled.
categorical_pipeline = build_logreg_pipeline(
    categorical_features=base_categorical_cols + repayment_cols,
    numerical_features=base_numerical_cols,
)

# Verify that both strategies use all 23 predictors exactly once.
for strategy, categorical_features, numerical_features in [
    ("Numerical", base_categorical_cols, base_numerical_cols + repayment_cols),
    ("Categorical", base_categorical_cols + repayment_cols, base_numerical_cols),
]:
    configured_features = categorical_features + numerical_features

    assert len(configured_features) == len(set(configured_features))
    expected_features = base_categorical_cols + repayment_cols + base_numerical_cols
    assert set(configured_features) == set(expected_features)

    print(
        f"{strategy}: "
        f"{len(categorical_features)} categorical + "
        f"{len(numerical_features)} numerical input columns"
    )

Numerical: 3 categorical + 20 numerical input columns
Categorical: 9 categorical + 14 numerical input columns


### Configuration check

Both configurations cover all 23 predictors exactly once. The numerical strategy assigns 3 inputs to One-Hot Encoding and 20 to scaling. The categorical strategy assigns 9 inputs to One-Hot Encoding and 14 to scaling. No transformer has been fitted outside cross-validation.

## Run the encoding comparison

Materialize the five StratifiedKFold index pairs once and pass the same list to both experiments. This guarantees that the two strategies are trained and validated on identical records with comparable target proportions.

Use `cross_validate` with ROC-AUC and return each fitted pipeline. The fitted models are inspected for convergence, and their preprocessors are inspected to measure the resulting feature dimensionality. Fit times are recorded as descriptive execution information. No hyperparameters are tuned in this experiment.

In [68]:
# Generate the fold indices once so both strategies use identical splits.
cv_splits = list(cv_splitter.split(X_full_train, y_full_train))

experiment_pipelines = {
    "Numerical": numerical_pipeline,
    "Categorical": categorical_pipeline,
}

experiment_outputs = {}
fold_result_frames = []
summary_rows = []

for strategy, experiment_pipeline in experiment_pipelines.items():
    # cross_validate clones the full pipeline for every fold.
    output = cross_validate(
        estimator=experiment_pipeline,
        X=X_full_train,
        y=y_full_train,
        cv=cv_splits,
        scoring="roc_auc",
        return_estimator=True,
        error_score="raise",
    )
    experiment_outputs[strategy] = output

    fold_scores = output["test_score"]
    fitted_pipelines = output["estimator"]

    # Access the Logistic Regression step inside every fitted pipeline.
    solver_iterations = [
        fitted_pipeline.named_steps["model"].n_iter_[0]
        for fitted_pipeline in fitted_pipelines
    ]

    # Count the transformed features created in every fold.
    transformed_dimensions = [
        len(
            fitted_pipeline
            .named_steps["preprocessing"]
            .get_feature_names_out()
        )
        for fitted_pipeline in fitted_pipelines
    ]

    fold_result_frames.append(pd.DataFrame({
        "strategy": strategy,
        "fold": range(1, len(cv_splits) + 1),
        "roc_auc": fold_scores,
        "fit_time_seconds": output["fit_time"],
        "iterations": solver_iterations,
        "transformed_features": transformed_dimensions,
    }))

    summary_rows.append({
        "Strategy": strategy,
        "Mean ROC-AUC": fold_scores.mean(),
        "Std ROC-AUC": fold_scores.std(ddof=1),
        "Mean fit time (s)": output["fit_time"].mean(),
        "Min transformed features": min(transformed_dimensions),
        "Max transformed features": max(transformed_dimensions),
    })

pay_encoding_fold_results = pd.concat(
    fold_result_frames,
    ignore_index=True,
)

pay_encoding_summary = (
    pd.DataFrame(summary_rows)
    .set_index("Strategy")
)

# Compare the two scores within the same fold.
paired_fold_comparison = (
    pay_encoding_fold_results
    .pivot(index="fold", columns="strategy", values="roc_auc")
)
paired_fold_comparison["Categorical - Numerical"] = (
    paired_fold_comparison["Categorical"]
    - paired_fold_comparison["Numerical"]
)

display(pay_encoding_fold_results)
display(pay_encoding_summary)
display(paired_fold_comparison)

# A value below 1,000 confirms that each solver finished before max_iter.
assert pay_encoding_fold_results["iterations"].max() < 1000

,strategy,fold,roc_auc,fit_time_seconds,iterations,transformed_features
0,Numerical,1,0.730490,0.344969,59,31
1,Numerical,2,0.723353,0.439551,64,31
2,Numerical,3,0.718163,0.343593,66,31
3,Numerical,4,0.725550,0.369854,62,31
4,Numerical,5,0.715010,0.527963,66,31
5,Categorical,1,0.780734,0.537899,72,89
6,Categorical,2,0.765065,0.437743,73,88
7,Categorical,3,0.770769,0.523173,89,87
8,Categorical,4,0.762378,0.686277,95,89
9,Categorical,5,0.762328,0.512463,84,89


,Mean ROC-AUC,Std ROC-AUC,Mean fit time (s),Min transformed features,Max transformed features
Strategy,,,,,
Numerical,0.722513,0.006099,0.405186,31,31
Categorical,0.768255,0.007776,0.539511,87,89


strategy,Categorical,Numerical,Categorical - Numerical
fold,,,
1,0.780734,0.730490,0.050244
2,0.765065,0.723353,0.041712
3,0.770769,0.718163,0.052605
4,0.762378,0.725550,0.036828
5,0.762328,0.715010,0.047318


### Encoding comparison results

The categorical strategy achieves a mean ROC-AUC of 0.7683 (standard deviation 0.0078), compared with 0.7225 (standard deviation 0.0061) for the numerical strategy. The mean improvement is 0.0457 ROC-AUC. Categorical encoding scores higher in all five paired folds; fold improvements range from 0.0368 to 0.0526. This consistent result supports the observed advantage, but the difference in cross-validation means alone is not a claim of statistical significance.

The numerical strategy produces 31 transformed features and assumes an ordered, approximately linear relationship between repayment codes and log-odds after scaling. The categorical strategy produces 87 to 89 transformed features, depending on which rare codes appear in a fitting fold. It assigns an independent coefficient to each observed code and month, which avoids imposing undocumented distances and makes code-specific effects inspectable, at the cost of a larger model.

Mean fit time is approximately 0.43 seconds for numerical encoding and 0.54 seconds for categorical encoding in this run. Every solver converges below `max_iter=1000`: numerical folds require 59 to 66 iterations, while categorical folds require 72 to 95. No convergence warning occurs. The separate pandas warning concerns the deprecated XLS reader and is unrelated to model fitting.

For Logistic Regression, use categorical repayment-status encoding as the preferred configuration because its improvement is substantial and consistent across identical folds, and because it avoids assigning unsupported distances to -2, -1, 0, and positive delay codes. Preserve the numerical configuration as a documented baseline and reconsider both representations for models with different inductive assumptions. The held-out test set remains untouched.